# Replication Notebook

Replicates the analysis: **RM-ANOVA, Linear Mixed Models (lme4-equivalent), and correlation plots** in Python.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.anova import AnovaRM
import statsmodels.formula.api as smf

sns.set_theme(style='whitegrid')
rng = np.random.default_rng(42)

## 2. Example data (replace with `pd.read_csv("your_data.csv")`)

In [ ]:
conds, times = ['A', 'B', 'C'], ['T1', 'T2', 'T3']
rows = []
for s in range(30):
    for c in conds:
        for t in times:
            mu = 10 + 2 * conds.index(c) + times.index(t)
            rows.append({'subject': f'S{s:02d}', 'condition': c, 'time': t,
                         'y': rng.normal(mu, 2)})
dat = pd.DataFrame(rows)
dat.head()

## 3. Repeated-measures ANOVA

In [ ]:
aov = AnovaRM(dat, depvar='y', subject='subject', within=['condition', 'time']).fit()
print(aov)

## 4. Linear Mixed Models (lme4 equivalent via statsmodels)

In [ ]:
# Random-intercept model: y ~ condition * time + (1 | subject)
m0 = smf.mixedlm('y ~ condition * time', dat, groups=dat['subject']).fit()
print(m0.summary())

In [ ]:
# Random-slope model: y ~ condition * time + (time | subject)
exog_re = pd.get_dummies(dat['time'], drop_first=True).astype(float)
m1 = smf.mixedlm('y ~ condition * time', dat, groups=dat['subject'], exog_re=exog_re).fit()
print(m1.summary())
print('\nRandom-effect covariance:\n', m1.cov_re)
print('Residual SD:', m1.scale ** 0.5)

In [ ]:
# Diagnostics: residuals vs fitted
plt.figure(figsize=(5, 4))
plt.scatter(m0.fittedvalues, m0.resid, alpha=0.6)
plt.axhline(0, color='red', ls='--')
plt.xlabel('Fitted'); plt.ylabel('Residual')
plt.title('Residuals vs Fitted (random-intercept model)')
plt.tight_layout(); plt.savefig('residuals_vs_fitted.png', dpi=150); plt.show()

## 5. Correlation plots

In [ ]:
subj_means = dat.groupby(['subject', 'condition'])['y'].mean().unstack()  # columns A, B, C
cor_mat = subj_means.corr()
print(cor_mat)

plt.figure(figsize=(5, 4))
sns.heatmap(cor_mat, annot=True, cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Correlation between conditions (subject means)')
plt.tight_layout(); plt.savefig('correlation_heatmap.png', dpi=150); plt.show()

In [ ]:
plt.figure(figsize=(5, 4))
sns.regplot(x=subj_means['A'], y=subj_means['B'])
r = subj_means['A'].corr(subj_means['B'])
plt.title(f'A vs B (r = {r:.2f})')
plt.tight_layout(); plt.savefig('scatter_A_B.png', dpi=150); plt.show()

## 6. Notes
- `AnovaRM` gives classic within-subject ANOVA; for Greenhouse-Geisser corrections use `pingouin.rm_anova`.
- statsmodels `mixedlm` approximates `lmer`; for full lme4 syntax run the companion `analysis_script.R` in R.